# 01 — Foundations
**Goal:** Understand matplotlib's object model and produce a clean, publication-quality single plot from scratch.

By the end of this notebook you will:
- Know the difference between the `pyplot` API and the object-oriented (OO) API
- Understand the Figure → Axes → Artist hierarchy
- Be able to control spines, grids, fonts, colors, and layout explicitly
- Transform a default matplotlib plot into a publication-ready figure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulated training data — used throughout this notebook
np.random.seed(42)
epochs     = np.arange(1, 51)
train_loss = 2.5 * np.exp(-0.08 * epochs) + 0.05 * np.random.randn(50)
val_loss   = 2.5 * np.exp(-0.07 * epochs) + 0.12 + 0.08 * np.random.randn(50)

---
## Part 1 — `pyplot` vs the Object-Oriented API

Most tutorials teach `plt.plot(...)`, `plt.xlabel(...)`, etc. This is the **pyplot API**. It works by maintaining a hidden "current figure" state, which causes bugs in multi-figure or reusable code.

Top research code uses the **OO API**: you explicitly create a `Figure` and `Axes` object and call methods on them.

The key line: `fig, ax = plt.subplots()`
- `fig` — the canvas (controls size, saving)
- `ax` — the plot area (controls data, labels, ticks, spines)

**Exercise:** Write the same plot twice — once with the pyplot API, once with the OO API.

In [ ]:
# pyplot API (just so you recognise it)
# TODO: plot train_loss vs epochs using plt.plot(), add a title, show it


In [ ]:
# OO API — use this going forward
# TODO: create fig and ax with plt.subplots(), plot train_loss vs epochs on ax, add a title, show it
# Hint: ax.plot(...), ax.set_title(...)


---
## Part 2 — The Figure → Axes → Artist hierarchy

Everything in matplotlib is an **Artist** — a Python object that knows how to draw itself.

```
Figure
└── Axes
    ├── Line2D          ← your plotted data
    ├── XAxis / YAxis
    │   ├── Tick objects
    │   └── Text labels
    ├── Title (Text)
    └── Spines          ← the four border lines
```

Understanding this means you can reach in and modify *any* element.

**Exercise:** Inspect the artists in a plot.

In [ ]:
fig, ax = plt.subplots()
line, = ax.plot(epochs, train_loss, label='Train loss')

# TODO: print the type of ax, line, and ax.xaxis
# TODO: print the keys of ax.spines (hint: ax.spines is a dict)

plt.close()   # close without rendering

---
## Part 3 — The default plot (and what's wrong with it)

**Exercise:** Make a basic training loss plot (train + val) with default settings — no customisation at all. Just get something on screen.

In [ ]:
# TODO: using the OO API, plot both train_loss and val_loss vs epochs
# Add x-label 'Epoch', y-label 'Loss', a title, and a legend
# Don't change any styling yet — we want to see the ugly default


**Look at the output and note the problems:**
- [ ] Figure size feels off for a paper column
- [ ] Font sizes are too small
- [ ] The 4-sided box frame adds visual noise
- [ ] No grid — hard to read values
- [ ] Default colors — not intentional
- [ ] Legend has a distracting box

You'll fix each one in the steps below.

---
## Part 4 — Fixing it, one step at a time

### Step 4a — Figure size

Paper column widths:
- **Single column:** ~3.5 in wide
- **Full width / double column:** ~7 in wide
- **Height:** typically 0.65–0.75× the width

Always set `figsize` explicitly. Never rely on the 6.4×4.8 default.

**Exercise:** Recreate the plot with an appropriate `figsize`.

In [ ]:
# TODO: pass figsize=(5, 3.5) to plt.subplots() and re-plot
# Hint: plt.subplots(figsize=(width, height))


### Step 4b — Font sizes

A common mistake: using default font sizes (10–12pt) then scaling the figure down — text becomes unreadable.

Rule of thumb:
- Axis labels: **12pt**
- Tick labels: **11pt**
- Legend: **10pt**
- Title (if used): **13pt**

**Exercise:** Add explicit font sizes to every text element.

In [ ]:
# TODO: reproduce the plot with figsize=(5, 3.5) and set explicit font sizes
# Hints:
#   ax.set_xlabel(..., fontsize=12)
#   ax.set_ylabel(..., fontsize=12)
#   ax.tick_params(labelsize=11)
#   ax.legend(fontsize=10)


### Step 4c — Spines

The default 4-sided box adds no information. Clean research figures remove the **top** and **right** spines, keeping only the axis lines.

This is the single biggest visual upgrade you can make.

**Exercise:** Remove the top and right spines.

In [ ]:
# TODO: build on the previous plot and hide the top and right spines
# Hint: ax.spines['top'].set_visible(False)


### Step 4d — Grid

A subtle horizontal-only grid helps readers trace values. Key rules:
- Horizontal only (not vertical)
- Light gray, thin
- Drawn *behind* data (use `zorder=0` and `set_axisbelow`)

**Exercise:** Add a minimal horizontal grid.

In [ ]:
# TODO: add a horizontal grid with color='#E0E0E0', linewidth=0.8
# Hint: ax.yaxis.grid(True, ...) and ax.set_axisbelow(True)


### Step 4e — Colors, line widths, and legend cleanup

Use explicit hex colors instead of defaults. For two related curves (train/val), a clear pattern:
- One solid, one dashed
- Contrasting but not jarring colors (e.g., blue + red)
- `linewidth=1.8` is a good starting point
- Legend: `frameon=False` always

**Exercise:** Assign intentional colors and line styles.

In [ ]:
# TODO: plot train in solid blue (#2563EB) and val in dashed red (#DC2626)
# Use linewidth=1.8 and frameon=False on the legend


### Step 4f — Layout padding

`fig.tight_layout()` prevents labels from being clipped when saving. Always call it before `plt.show()` or `savefig()`.

**Exercise:** Put the full polished plot together in one cell.

In [ ]:
# TODO: write the complete clean figure in one cell — all steps combined
# figsize, font sizes, spines, grid, colors, legend, tight_layout


---
## Part 5 — Before / After

**Exercise:** Use `plt.subplots(1, 2, figsize=(11, 3.5))` to put the default plot (left) and your polished plot (right) side by side.

In [ ]:
# TODO: create a 1×2 subplot figure
# Left ax: the raw default plot
# Right ax: the fully polished plot
# Hint: fig, (ax_before, ax_after) = plt.subplots(1, 2, figsize=(11, 3.5))


---
## Part 6 — Reusable helper

Rather than repeating the cleanup code in every figure, write a small helper function `clean_axes(ax)` that applies all the style fixes.

This seeds the style system you'll build properly in notebook 02.

**Exercise:** Write and test the helper.

In [ ]:
def clean_axes(ax):
    """Apply minimal publication-style cleanup to an Axes object."""
    # TODO: hide top and right spines
    # TODO: add horizontal grid
    # TODO: set tick label size to 11
    # TODO: call set_axisbelow(True)
    return ax

In [ ]:
# TODO: verify the helper works — reproduce the final polished plot using clean_axes(ax)


---
## Summary

| Concept | What to remember |
|---------|------------------|
| OO API | Always `fig, ax = plt.subplots()` — never raw `plt.plot()` in final figures |
| `figsize` | Set explicitly — `(5, 3.5)` for single column |
| Font sizes | 12pt labels · 11pt ticks · 10pt legend |
| Spines | Remove top + right |
| Grid | Horizontal only, light gray, `set_axisbelow(True)` |
| Legend | `frameon=False` always |
| Layout | `fig.tight_layout()` before every save/show |

**Next:** `02_style_system.ipynb` — turning all of this into a reusable `rcParams` style sheet so it's automatic across your whole paper.